# nnU-Net BraTS 2024 MEN-RT — Google Colab Pipeline

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run cells top to bottom in order
3. Keep browser tab open during training (3-4 hours)

**Edit ONLY Cell 3 (config). All other cells run as-is.**

| Cell | Step | Est. Time |
|---|---|---|
| 1 | Install packages | 3-5 min |
| 2 | GPU check | 10 sec |
| 3 | Config (edit here) | 5 sec |
| 4 | Google Drive mount | 30 sec |
| 5 | Kaggle credentials | 30 sec |
| 6 | Download BraTS data | 1-2 min |
| 7 | Clone repository | 1 min |
| 8 | Register custom trainer | 10 sec |
| 9 | Prepare dataset | 2-4 min |
| 10 | Verify dataset | 10 sec |
| 11 | Preprocessing | 5-10 min |
| 12 | **Train all 5 folds** | **3-4 hours** |
| 13 | Training history | 10 sec |
| 14 | Verify checkpoints | 10 sec |
| 15 | Inference | 30-60 min |
| 16 | Post-processing | 10 min |
| 17 | Evaluation | 10 min |
| 18 | Visualizations | 5 min |
| 19 | Show results | 10 sec |

In [ ]:
# ── Cell 1: Install All Packages ─────────────────────────────────────────────
# Run this FIRST before any other cell. Takes 3-5 minutes.
import subprocess, sys

packages = [
    'nnunetv2>=2.4',
    'nibabel>=5.0',
    'SimpleITK>=2.3',
    'scipy>=1.11',
    'scikit-image>=0.21',
    'medpy>=0.5',
    'surface-distance>=0.1',
    'pandas>=2.0',
    'matplotlib>=3.8',
    'seaborn>=0.13',
    'mlflow>=2.10',
    'python-dotenv>=1.0',
    'pyyaml>=6.0',
    'tqdm>=4.66',
    'loguru>=0.7',
    'rich>=13.0',
    'kaggle',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    check=True
)
print('All packages installed successfully.')

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────────
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if not torch.cuda.is_available():
    raise RuntimeError('ERROR: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('GPU OK.')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
# EDIT ONLY THIS CELL. All other cells read from these variables.

# ── Kaggle dataset slug (from kaggle.com/datasets/USERNAME/DATASETNAME) ───────
KAGGLE_DATASET = 'maryampervaiz24029/meningioma-50-cases'

# ── GitHub repository URL (must be public) ────────────────────────────────────
GITHUB_REPO = 'https://github.com/maryampervaiz-alt/nnunet-training.git'

# ── Paths (do not change unless you know what you are doing) ──────────────────
REPO_DIR     = '/content/nnunet'
DATA_DIR     = '/content/brats_data/BraTS-MEN-RT-Train-v2'
DRIVE_OUTPUT = '/content/drive/MyDrive/nnunet_checkpoints'

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_ID       = '001'
DATASET_NAME     = 'BraTSMENRT'
LABEL_SUFFIX     = 'gtv'
MAX_TRAIN_CASES  = 20

# ── Training ──────────────────────────────────────────────────────────────────
NNUNET_CONFIGURATION = '3d_fullres'
NNUNET_TRAINER_CLASS = 'nnUNetTrainerEarlyStopping'
NNUNET_PLANS         = 'nnUNetPlans'
NUM_FOLDS            = 5
NNUNET_SEED          = 42
NNUNET_NUM_EPOCHS    = 10    # 10=pilot, 50=preliminary, 1000=full
RESUME_TARGET_EPOCHS = 50

# ── Early stopping (no effect when NNUNET_NUM_EPOCHS <= ES_WARMUP) ────────────
ES_PATIENCE  = 50
ES_MIN_DELTA = 0.0001
ES_WARMUP    = 50

# ── Hardware ──────────────────────────────────────────────────────────────────
CUDA_VISIBLE_DEVICES = '0'
NNUNET_N_PROC_DA     = '2'      # reduce if training hangs
NNUNET_COMPILE       = 'false'  # avoids 30-min JIT overhead

# ── Inference ─────────────────────────────────────────────────────────────────
INFERENCE_STEP_SIZE   = 0.5
INFERENCE_DISABLE_TTA = 'false'
INFERENCE_DEVICE      = 'cuda'

# ── Evaluation ────────────────────────────────────────────────────────────────
PP_THRESHOLDS      = '10,25,50,100,200,500'
NSD_TOLERANCE_MM   = 2.0
LOW_DICE_THRESHOLD = 0.5
BOOTSTRAP_N        = 2000

print('Configuration set. Do not run this cell again unless you want to change settings.')

In [ ]:
# ── Cell 4: Mount Google Drive ────────────────────────────────────────────────
# Checkpoints are saved here after every fold so training can be resumed.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted.')
print(f'Checkpoints will be saved to: {DRIVE_OUTPUT}')

In [ ]:
# ── Cell 5: Kaggle Credentials ────────────────────────────────────────────────
# Option A (recommended): Add kaggle.json content as Colab Secret named 'kaggle_json'
#   Left sidebar → key icon → Add new secret → Name: kaggle_json
# Option B: Upload kaggle.json file when prompted

import os, json, shutil
from pathlib import Path

os.makedirs('/root/.kaggle', exist_ok=True)

try:
    from google.colab import userdata
    kaggle_content = userdata.get('kaggle_json')
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_content)
    print('Kaggle credentials loaded from Colab secrets.')
except Exception:
    from google.colab import files
    print('Upload your kaggle.json file (from kaggle.com → Settings → API → Create Legacy API Key):')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.copy(fname, '/root/.kaggle/kaggle.json')
    print(f'Credentials loaded from uploaded file: {fname}')

os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify
with open('/root/.kaggle/kaggle.json') as f:
    info = json.load(f)
print(f'Kaggle username: {info["username"]}')
print('Kaggle credentials OK.')

In [ ]:
# ── Cell 6: Download BraTS Data ───────────────────────────────────────────────
import subprocess
from pathlib import Path

Path('/content/brats_data').mkdir(parents=True, exist_ok=True)

if Path(DATA_DIR).exists():
    cases = list(Path(DATA_DIR).iterdir())
    print(f'Data already exists: {len(cases)} items in {DATA_DIR}')
else:
    print(f'Downloading {KAGGLE_DATASET} ...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', KAGGLE_DATASET,
         '-p', '/content/brats_data', '--unzip'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
        raise RuntimeError('Data download failed.')

# Verify
if not Path(DATA_DIR).exists():
    print('Contents of /content/brats_data:')
    for p in Path('/content/brats_data').iterdir():
        print(f'  {p}')
    raise RuntimeError(
        f'Expected data at {DATA_DIR} not found.\n'
        'Update DATA_DIR in Cell 3 to match the path shown above.'
    )

cases = sorted(Path(DATA_DIR).iterdir())
print(f'Data ready: {len(cases)} cases at {DATA_DIR}')

In [ ]:
# ── Cell 7: Clone Repository ──────────────────────────────────────────────────
import subprocess, shutil, sys
from pathlib import Path

# Clean re-clone every time to ensure fresh code
# (nnunet_raw / preprocessed / results are stored OUTSIDE repo dir)
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
    print('Removed existing repo directory.')

result = subprocess.run(
    ['git', 'clone', GITHUB_REPO, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERROR:', result.stderr)
    raise RuntimeError('Git clone failed. Make sure the repo is public.')
print('Repository cloned.')

# Set environment variables
import os
os.environ.update({
    'nnUNet_raw'             : '/content/nnunet_work/nnunet_raw',
    'nnUNet_preprocessed'    : '/content/nnunet_work/nnunet_preprocessed',
    'nnUNet_results'         : '/content/nnunet_work/nnunet_results',
    'DATASET_ID'             : DATASET_ID,
    'DATASET_NAME'           : DATASET_NAME,
    'LABEL_SUFFIX'           : LABEL_SUFFIX,
    'MAX_TRAIN_CASES'        : str(MAX_TRAIN_CASES),
    'TRAIN_SOURCE'           : DATA_DIR,
    'VAL_SOURCE'             : '',
    'NNUNET_CONFIGURATION'   : NNUNET_CONFIGURATION,
    'NNUNET_TRAINER_CLASS'   : NNUNET_TRAINER_CLASS,
    'NNUNET_PLANS_IDENTIFIER': NNUNET_PLANS,
    'NUM_FOLDS'              : str(NUM_FOLDS),
    'NNUNET_SEED'            : str(NNUNET_SEED),
    'NNUNET_NUM_EPOCHS'      : str(NNUNET_NUM_EPOCHS),
    'RESUME_TARGET_EPOCHS'   : str(RESUME_TARGET_EPOCHS),
    'ES_PATIENCE'            : str(ES_PATIENCE),
    'ES_MIN_DELTA'           : str(ES_MIN_DELTA),
    'ES_WARMUP'              : str(ES_WARMUP),
    'INFERENCE_STEP_SIZE'    : str(INFERENCE_STEP_SIZE),
    'INFERENCE_DISABLE_TTA'  : INFERENCE_DISABLE_TTA,
    'INFERENCE_DEVICE'       : INFERENCE_DEVICE,
    'PP_THRESHOLDS'          : PP_THRESHOLDS,
    'NSD_TOLERANCE_MM'       : str(NSD_TOLERANCE_MM),
    'LOW_DICE_THRESHOLD'     : str(LOW_DICE_THRESHOLD),
    'BOOTSTRAP_N'            : str(BOOTSTRAP_N),
    'CUDA_VISIBLE_DEVICES'   : CUDA_VISIBLE_DEVICES,
    'nnUNet_n_proc_DA'       : NNUNET_N_PROC_DA,
    'nnUNet_compile'         : NNUNET_COMPILE,
    'EXPERIMENT_NAME'        : 'BraTSMENRT_baseline',
    'MLFLOW_TRACKING_URI'    : '/content/nnunet_work/experiments/mlruns',
    'KAGGLE_OUTPUT_DIR'      : DRIVE_OUTPUT,
    'PYTHONUNBUFFERED'       : '1',
})

# Create all working directories
for d in [
    '/content/nnunet_work/nnunet_raw',
    '/content/nnunet_work/nnunet_preprocessed',
    '/content/nnunet_work/nnunet_results',
    '/content/nnunet_work/metrics',
    '/content/nnunet_work/results',
    '/content/nnunet_work/visualizations',
    '/content/nnunet_work/inference_outputs',
    '/content/nnunet_work/logs',
    '/content/nnunet_work/checkpoints',
]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Set working directory and Python path
os.chdir(REPO_DIR)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')
print('Environment configured.')

In [ ]:
# ── Cell 8: Register Custom nnU-Net Trainer ───────────────────────────────────
import shutil, importlib.util, os
from pathlib import Path
import nnunetv2

trainer_src = Path(REPO_DIR) / 'src' / 'training' / 'nnunet_trainer_es.py'
if not trainer_src.exists():
    raise FileNotFoundError(f'Custom trainer not found: {trainer_src}')

nnunet_pkg   = Path(nnunetv2.__file__).parent
trainer_dst  = nnunet_pkg / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'
shutil.copy2(trainer_src, trainer_dst)

# Verify the class is importable
spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
if not hasattr(mod, 'nnUNetTrainerEarlyStopping'):
    raise ImportError('nnUNetTrainerEarlyStopping class not found in trainer file')

print(f'Custom trainer registered at: {trainer_dst}')
print('Trainer registration OK.')

In [ ]:
# ── Cell 9: STEP 1 — Prepare Dataset ─────────────────────────────────────────
# Converts BraTS data to nnU-Net format. ~2-4 minutes.
import subprocess, sys, os
from pathlib import Path

print(f'Source : {os.environ["TRAIN_SOURCE"]}')
print(f'Output : {os.environ["nnUNet_raw"]}')
print(f'Cases  : {os.environ["MAX_TRAIN_CASES"]}')
print()

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/01_prepare_dataset.py',
     '--train',     os.environ['TRAIN_SOURCE'],
     '--skip-val',
     '--log-dir',   '/content/nnunet_work/logs',
     '--max-cases', os.environ['MAX_TRAIN_CASES']],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Dataset preparation failed (rc={proc.returncode})')
print('\nDataset preparation complete.')

In [ ]:
# ── Cell 10: Verify Dataset Structure ────────────────────────────────────────
import json, os
from pathlib import Path

ds_dir = Path(os.environ['nnUNet_raw']) / f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
images = sorted((ds_dir / 'imagesTr').glob('*.nii.gz'))
labels = sorted((ds_dir / 'labelsTr').glob('*.nii.gz'))

print(f'Dataset dir : {ds_dir}')
print(f'Images      : {len(images)}')
print(f'Labels      : {len(labels)}')

if len(images) == 0:
    raise RuntimeError('No images found — check Cell 9 output for errors.')
if len(images) != len(labels):
    raise RuntimeError(f'Image/label count mismatch: {len(images)} vs {len(labels)}')

ds_json = ds_dir / 'dataset.json'
if ds_json.exists():
    d = json.loads(ds_json.read_text())
    print(f'dataset.json: {d.get("numTraining")} training cases')

print('Dataset verification OK.')

In [ ]:
# ── Cell 11: STEP 2 — nnU-Net Planning & Preprocessing ───────────────────────
# Runs nnUNetv2_plan_and_preprocess. ~5-10 minutes for 20 cases.
# --np 1 = 1 CPU worker to avoid Colab RAM crash (does NOT affect model config)
import subprocess, sys, os

print('Running nnUNetv2_plan_and_preprocess (1 worker to avoid RAM crash)...')
proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py',
     '--np', '1',
     '--log-dir', '/content/nnunet_work/logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Preprocessing failed (rc={proc.returncode})')
print('\nPreprocessing complete.')

In [ ]:
# ── Cell 12: STEP 3 — Train All Folds ────────────────────────────────────────
# ~3-4 hours total. Checkpoints saved to Google Drive after every fold.
# If interrupted: re-run this cell — completed folds are skipped automatically.

import os, subprocess, sys, shutil
from pathlib import Path

NUM_FOLDS    = int(os.environ['NUM_FOLDS'])
NUM_EPOCHS   = int(os.environ['NNUNET_NUM_EPOCHS'])
TRAINER      = os.environ['NNUNET_TRAINER_CLASS']
PLANS        = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG       = os.environ['NNUNET_CONFIGURATION']
DS           = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
RESULTS_DIR  = Path(os.environ['nnUNet_results'])
METRICS_DIR  = Path('/content/nnunet_work/metrics')

def _ckpt_dir(fold):
    return RESULTS_DIR / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'

def _is_done(fold):
    return (_ckpt_dir(fold) / 'checkpoint_final.pth').exists()

def _save_fold_to_drive(fold):
    """Copy fold checkpoints + metrics to Google Drive immediately after training."""
    drive_nnunet = Path(DRIVE_OUTPUT) / 'nnunet_results'
    src = _ckpt_dir(fold)
    dst = drive_nnunet / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    dst.mkdir(parents=True, exist_ok=True)
    # Copy checkpoints
    for fp in list(src.glob('*.pth')) + list(src.glob('training_log*.txt')):
        shutil.copy2(fp, dst / fp.name)
    # Copy metrics CSV
    mcsv = METRICS_DIR / f'fold_{fold}_training.csv'
    if mcsv.exists():
        drive_metrics = Path(DRIVE_OUTPUT) / 'metrics'
        drive_metrics.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mcsv, drive_metrics / mcsv.name)
    n = sum(1 for _ in dst.rglob('*.pth'))
    print(f'  [fold {fold}] {n} checkpoint files saved to Drive: {dst}')

def _restore_fold_from_drive(fold):
    """Restore fold checkpoint from Drive if it exists (for resume)."""
    drive_fold = Path(DRIVE_OUTPUT) / 'nnunet_results' / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    if not drive_fold.exists():
        return False
    dst = _ckpt_dir(fold)
    dst.mkdir(parents=True, exist_ok=True)
    for fp in drive_fold.glob('*.pth'):
        shutil.copy2(fp, dst / fp.name)
    print(f'  [fold {fold}] Restored from Drive: {list(drive_fold.glob("*.pth"))}')
    return True

# ── Restore any completed folds from Drive before starting ───────────────────
print('Checking Drive for existing checkpoints ...')
for fold in range(NUM_FOLDS):
    if not _is_done(fold):
        _restore_fold_from_drive(fold)

print(f'\nTraining {NUM_FOLDS} folds x {NUM_EPOCHS} epochs')
print(f'Trainer : {TRAINER}')
print(f'Output  : {DRIVE_OUTPUT}')
fold_results = {}

for fold in range(NUM_FOLDS):
    print(f'\n{"="*64}')
    if _is_done(fold):
        print(f'  Fold {fold}: SKIPPED (checkpoint_final.pth exists)')
        fold_results[fold] = 'skipped'
        continue

    ckpt_exists = (_ckpt_dir(fold) / 'checkpoint_latest.pth').exists()
    action = 'Resuming' if ckpt_exists else 'Training'
    print(f'  {action} fold {fold}/{NUM_FOLDS-1} — {NUM_EPOCHS} epochs')
    print(f'  Checkpoints: {_ckpt_dir(fold)}')
    print('='*64)

    cmd = [
        sys.executable, '-u', 'scripts/03_train.py',
        '--folds',           str(fold),
        '--num-epochs',      str(NUM_EPOCHS),
        '--seed',            os.environ['NNUNET_SEED'],
        '--trainer',         TRAINER,
        '--plans',           PLANS,
        '--es-patience',     os.environ['ES_PATIENCE'],
        '--es-min-delta',    os.environ['ES_MIN_DELTA'],
        '--es-warmup',       os.environ['ES_WARMUP'],
        '--log-dir',         '/content/nnunet_work/logs',
        '--metrics-dir',     '/content/nnunet_work/metrics',
        '--checkpoints-dir', '/content/nnunet_work/checkpoints',
    ]
    if ckpt_exists:
        cmd.append('--continue-training')

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    status = 'OK' if proc.returncode == 0 else f'FAILED (rc={proc.returncode})'
    fold_results[fold] = status
    print(f'\nFold {fold}: {status}')
    _save_fold_to_drive(fold)

print(f'\n{"="*64}\nTraining summary:')
for fold, status in fold_results.items():
    print(f'  Fold {fold}: {status}')
print(f'{"="*64}')
print(f'All checkpoints saved to: {DRIVE_OUTPUT}')

In [ ]:
# ── Cell 13: Training History — All Folds ─────────────────────────────────────
import pandas as pd, os
from pathlib import Path

NUM_FOLDS   = int(os.environ['NUM_FOLDS'])
METRICS_DIR = Path('/content/nnunet_work/metrics')

print('Per-fold training summary:')
print('=' * 72)
print(f"  {'Fold':>4}  {'Epochs':>6}  {'Best DSC':>10}  {'Final DSC':>10}  {'Final Loss':>11}")
print('  ' + '-' * 55)

for fold in range(NUM_FOLDS):
    csv = METRICS_DIR / f'fold_{fold}_training.csv'
    if not csv.exists():
        print(f'  {fold:>4}  not trained yet')
        continue
    hist = pd.read_csv(csv)
    n_ep       = len(hist)
    best_dice  = hist['val_dice'].max()     if 'val_dice'   in hist.columns else float('nan')
    final_dice = hist['val_dice'].iloc[-1]  if 'val_dice'   in hist.columns else float('nan')
    final_loss = hist['train_loss'].iloc[-1] if 'train_loss' in hist.columns else float('nan')
    print(f'  {fold:>4}  {n_ep:>6}  {best_dice:>10.4f}  {final_dice:>10.4f}  {final_loss:>11.4f}')

print('=' * 72)

In [ ]:
# ── Cell 14: Verify Checkpoints ───────────────────────────────────────────────
import os
from pathlib import Path

TRAINER     = os.environ['NNUNET_TRAINER_CLASS']
PLANS       = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG      = os.environ['NNUNET_CONFIGURATION']
DS          = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
NUM_FOLDS   = int(os.environ['NUM_FOLDS'])
results_dir = Path(os.environ['nnUNet_results'])

print('Checkpoint verification:')
print('=' * 64)
all_ok = True
for fold in range(NUM_FOLDS):
    ckpt_dir = results_dir / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    if not ckpt_dir.exists():
        print(f'  Fold {fold}: NOT TRAINED')
        all_ok = False
        continue
    final  = ckpt_dir / 'checkpoint_final.pth'
    best   = ckpt_dir / 'checkpoint_best.pth'
    latest = ckpt_dir / 'checkpoint_latest.pth'
    f_sz = f'{final.stat().st_size/1e6:.1f} MB'  if final.exists()  else 'MISSING'
    b_sz = f'{best.stat().st_size/1e6:.1f} MB'   if best.exists()   else 'MISSING'
    l_sz = f'{latest.stat().st_size/1e6:.1f} MB' if latest.exists() else 'MISSING'
    status = 'OK' if final.exists() and best.exists() else 'PARTIAL'
    if status != 'OK': all_ok = False
    print(f'  Fold {fold} [{status}]  final={f_sz}  best={b_sz}  latest={l_sz}')
print('=' * 64)
print('All folds OK.' if all_ok else 'Some folds incomplete — see above.')

In [ ]:
# ── Cell 15: STEP 4 — Inference ───────────────────────────────────────────────
# Runs per-fold validation inference on held-out cases.
import os, subprocess, sys
from pathlib import Path

TRAINER   = os.environ['NNUNET_TRAINER_CLASS']
PLANS     = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG    = os.environ['NNUNET_CONFIGURATION']
DS        = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
NUM_FOLDS = int(os.environ['NUM_FOLDS'])

def _has_checkpoint(fold):
    d = Path(os.environ['nnUNet_results']) / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    return (d / 'checkpoint_final.pth').exists() or (d / 'checkpoint_best.pth').exists()

trained_folds = [f for f in range(NUM_FOLDS) if _has_checkpoint(f)]
print(f'Folds with checkpoints: {trained_folds}')
if not trained_folds:
    raise RuntimeError('No trained folds found. Run Cell 12 first.')

for fold in trained_folds:
    out_dir = f'/content/nnunet_work/inference_outputs/cv'
    print(f'\n{"="*64}\nInference fold {fold}\n{"="*64}')
    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/04_inference.py',
         '--cv-mode',
         '--folds',  str(fold),
         '--output', out_dir,
         '--log-dir', '/content/nnunet_work/logs'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'WARNING: Fold {fold} inference returned rc={proc.returncode}')
    else:
        preds = list(Path(f'{out_dir}/fold_{fold}').glob('*.nii.gz'))
        print(f'Fold {fold}: {len(preds)} predictions')

print('\nInference complete.')

In [ ]:
# ── Cell 16: Post-processing — CC Threshold Sweep ─────────────────────────────
import numpy as np, nibabel as nib, os, pandas as pd
from scipy import ndimage
from pathlib import Path

THRESHOLDS = [int(x) for x in os.environ['PP_THRESHOLDS'].split(',')]
NUM_FOLDS  = int(os.environ['NUM_FOLDS'])
gt_dir     = (Path(os.environ['nnUNet_raw'])
               / f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
               / 'labelsTr')
pred_root  = Path('/content/nnunet_work/inference_outputs/cv')

def cc_filter(arr, min_voxels):
    labeled, n = ndimage.label(arr > 0)
    if n == 0: return arr.copy().astype(np.uint8)
    sizes = ndimage.sum(arr > 0, labeled, range(1, n + 1))
    keep  = np.zeros_like(labeled, dtype=bool)
    for idx, sz in enumerate(sizes, 1):
        if sz >= min_voxels: keep |= (labeled == idx)
    return keep.astype(np.uint8)

def mean_dice(pred_dir, gt_dir):
    preds = sorted(Path(pred_dir).glob('*.nii.gz'))
    if not preds: return float('nan')
    dices = []
    for pp in preds:
        gp = Path(gt_dir) / pp.name
        if not gp.exists(): continue
        p = np.asarray(nib.load(pp).dataobj, dtype=np.uint8)
        g = np.asarray(nib.load(gp).dataobj, dtype=np.uint8)
        tp    = int(np.logical_and(p, g).sum())
        denom = int(p.sum()) + int(g.sum())
        dices.append(2 * tp / denom if denom > 0 else 1.0)
    return float(np.mean(dices)) if dices else float('nan')

trained_folds = [f for f in range(NUM_FOLDS) if (pred_root / f'fold_{f}').exists()]
print(f'Folds: {trained_folds}  |  Thresholds: {THRESHOLDS}')

sweep = []
for thresh in THRESHOLDS:
    dice_vals = []
    for fold in trained_folds:
        dst = pred_root / f'fold_{fold}_pp_{thresh}'
        dst.mkdir(parents=True, exist_ok=True)
        for pp in (pred_root / f'fold_{fold}').glob('*.nii.gz'):
            img     = nib.load(pp)
            cleaned = cc_filter(np.asarray(img.dataobj, dtype=np.uint8), thresh)
            nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
        dice_vals.append(mean_dice(dst, gt_dir))
    mean_d = float(np.nanmean(dice_vals))
    sweep.append({'min_voxels': thresh, 'mean_DSC': mean_d})
    print(f'  thresh={thresh:>5}  mean_DSC={mean_d:.4f}')

sweep_df    = pd.DataFrame(sweep)
BEST_THRESH = int(sweep_df.loc[sweep_df['mean_DSC'].idxmax(), 'min_voxels'])
print(f'\nBest threshold: {BEST_THRESH} voxels')

for fold in trained_folds:
    dst = pred_root / f'fold_{fold}_postproc'
    dst.mkdir(parents=True, exist_ok=True)
    n_changed = 0
    preds = list((pred_root / f'fold_{fold}').glob('*.nii.gz'))
    for pp in preds:
        img     = nib.load(pp)
        arr     = np.asarray(img.dataobj, dtype=np.uint8)
        cleaned = cc_filter(arr, BEST_THRESH)
        if not np.array_equal(arr, cleaned): n_changed += 1
        nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
    print(f'Fold {fold}: {n_changed}/{len(preds)} predictions modified by post-processing')

print('\nPost-processing complete.')

In [ ]:
# ── Cell 17: STEP 5 — Evaluation ──────────────────────────────────────────────
# Computes DSC, HD95, HD, NSD, precision, recall, specificity, volume metrics.
import subprocess, sys, os
from pathlib import Path

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/05_evaluate.py',
     '--cv-mode',
     '--pred',         '/content/nnunet_work/inference_outputs/cv',
     '--results-dir',  '/content/nnunet_work/results',
     '--nsd-tolerance', os.environ['NSD_TOLERANCE_MM'],
     '--low-dice',      os.environ['LOW_DICE_THRESHOLD'],
     '--bootstrap-n',   os.environ['BOOTSTRAP_N'],
     '--show-rankings',
     '--latex',
     '--stat-test',
     '--log-dir', '/content/nnunet_work/logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

results_dir = Path('/content/nnunet_work/results')
print('\nOutput files:')
for f in sorted(results_dir.glob('*.csv')):
    print(f'  {f.name}')
for f in sorted(results_dir.glob('*.tex')):
    print(f'  {f.name}')

# Save results to Drive
import shutil
drive_results = Path(DRIVE_OUTPUT) / 'results'
if results_dir.exists():
    shutil.copytree(str(results_dir), str(drive_results), dirs_exist_ok=True)
    print(f'\nResults saved to Drive: {drive_results}')

In [ ]:
# ── Cell 18: STEP 6 — Visualizations ─────────────────────────────────────────
import subprocess, sys, os, shutil
from pathlib import Path

subprocess.run(
    [sys.executable, 'scripts/06_visualize.py',
     '--all', '--cv-mode',
     '--pred-dir',    '/content/nnunet_work/inference_outputs/cv',
     '--results-dir', '/content/nnunet_work/results',
     '--metrics-dir', '/content/nnunet_work/metrics',
     '--output-dir',  '/content/nnunet_work/visualizations',
     '--n-cases', '15',
     '--log-dir', '/content/nnunet_work/logs'],
    env=os.environ.copy(),
)

viz_dir = Path('/content/nnunet_work/visualizations')
pngs = sorted(viz_dir.rglob('*.png'))
print(f'{len(pngs)} visualization files:')
for p in pngs:
    print(f'  {p.relative_to(viz_dir)}  ({p.stat().st_size//1024} KB)')

# Save visualizations to Drive
drive_viz = Path(DRIVE_OUTPUT) / 'visualizations'
if viz_dir.exists():
    shutil.copytree(str(viz_dir), str(drive_viz), dirs_exist_ok=True)
    print(f'\nVisualizations saved to Drive: {drive_viz}')

In [ ]:
# ── Cell 19: Results Table + Show Plots ──────────────────────────────────────
import pandas as pd, numpy as np, os
from pathlib import Path
from IPython.display import Image, display

NUM_FOLDS   = int(os.environ['NUM_FOLDS'])
results_dir = Path('/content/nnunet_work/results')
viz_dir     = Path('/content/nnunet_work/visualizations')

_METRICS = {
    'dice': 'DSC', 'hd95': 'HD95 (mm)', 'hd': 'HD (mm)',
    'nsd': 'NSD', 'precision': 'Precision', 'recall': 'Recall',
    'specificity': 'Specificity', 'volume_similarity': 'Vol. Similarity',
}

def show_table(csv_path, title):
    p = Path(csv_path)
    if not p.exists():
        print(f'{title}: not found ({p})')
        return
    df = pd.read_csv(p)
    rows = []
    for col, label in _METRICS.items():
        if col not in df.columns: continue
        v = pd.to_numeric(df[col], errors='coerce').replace([float('inf'), float('-inf')], float('nan')).dropna()
        if v.empty: continue
        rows.append({
            'Metric' : label,
            'Mean'   : f'{v.mean():.4f}',
            'Std'    : f'{v.std():.4f}',
            'Median' : f'{v.median():.4f}',
            'Min'    : f'{v.min():.4f}',
            'Max'    : f'{v.max():.4f}',
        })
    if not rows:
        print(f'{title}: no metric columns found')
        return
    print(f'\n{title} (n={len(df)} cases)')
    print('=' * 70)
    print(pd.DataFrame(rows).set_index('Metric').to_string())
    print('=' * 70)

# Per-fold results
for fold in range(NUM_FOLDS):
    show_table(results_dir / f'fold_{fold}_per_case.csv', f'Fold {fold} Validation')

# Combined CV results
show_table(results_dir / 'cv_combined.csv', 'ALL FOLDS COMBINED (Cross-Validation)')

# Wilcoxon test
wil = results_dir / 'cv_wilcoxon.csv'
if wil.exists():
    print('\nWilcoxon Signed-Rank Test: raw vs post-processed')
    print(pd.read_csv(wil).to_string(index=False))

# Show visualization plots
for plot in ['metrics_violin.png', 'metrics_boxplot.png',
             'fold_comparison.png', 'volume_scatter.png']:
    p = viz_dir / plot
    if p.exists():
        print(f'\n{plot}')
        display(Image(str(p), width=900))

# Training curves
for fold in range(NUM_FOLDS):
    p = viz_dir / f'training_curve_fold_{fold}.png'
    if p.exists():
        print(f'\nTraining curve — Fold {fold}')
        display(Image(str(p), width=800))

# Segmentation overlays (best, median, worst cases)
overlay_dir = viz_dir / 'overlays'
if overlay_dir.exists():
    overlays = sorted(overlay_dir.glob('*.png'))[:6]
    for img_path in overlays:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))

---
## Resume Instructions

If training was interrupted (GPU limit, session timeout, etc.):

1. Start a new Colab session with T4 GPU
2. Run **Cell 1** (install packages)
3. Run **Cell 2** (GPU check)
4. Run **Cell 3** (config)
5. Run **Cell 4** (mount Drive — this restores your checkpoints)
6. Run **Cell 5** (Kaggle credentials)
7. Run **Cell 6** (download data — fast, 1-2 min)
8. Run **Cell 7** (clone repo + set env)
9. Run **Cell 8** (register trainer)
10. **Skip Cells 9-11** (data already prepared — skip if `nnunet_raw` exists in Drive)
11. Run **Cell 12** — completed folds are automatically skipped, training resumes from checkpoint

**All checkpoints, metrics, results, and visualizations are saved to:**
`Google Drive → My Drive → nnunet_checkpoints`